# Milo voice render — Chatterbox on a free GPU (Kaggle / Colab)

**One account is enough.** Set the accelerator, **Run All**, wait for the zip, hand it over to be
merged and committed, then Run All again — the clone picks up the merged clips and skips them, so
every run continues where the last one stopped. `PLAN` is the order the bands are rendered in;
the loop moves to the next band by itself when one is complete.

Kaggle: Settings → Accelerator → **GPU T4 x2** (or P100), Internet **On**. Sessions are capped at 12 h
and the account at 30 GPU-h a week; `HOURS` stops cleanly before the session limit, and the zip is
refreshed after every chunk so nothing is lost if the session dies first.

⚠️ Merge each zip BEFORE the next run — otherwise the next run re-renders what the last zip already holds.

In [ ]:
PLAN  = [('9-11', ''), ('6-8', ''), ('teen', '12-14'), ('teen', '17-18'), ('teen', '15-16')]   # (corpus, band) in order
VOICE = 'IvUJKFyjVb5hItY9dJAT'   # Stevie. Teddy (3-5) is XjGYkUkzth8BPs29fmcV — already complete.
HOURS = 11          # stop cleanly before the session limit
CHUNK = 100         # lines per process; the zip is refreshed after every chunk

In [ ]:
import os, subprocess, sys, time, pathlib, shutil
WORK = pathlib.Path('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
os.chdir(WORK)
# Colab: a free runtime can be recycled at any moment and /content is wiped with it — keep the zips on Drive.
ZIP_DIR = WORK
if not os.path.isdir('/kaggle/working'):
    try:
        from google.colab import drive; drive.mount('/content/drive')
        ZIP_DIR = pathlib.Path('/content/drive/MyDrive/milo-voice'); ZIP_DIR.mkdir(parents=True, exist_ok=True)
        print('zips go to Google Drive:', ZIP_DIR)
    except Exception as e: print('no Drive mount, zips stay in', WORK, '—', e)
if not (WORK / 'learn').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RadlorInc/learn.git'], check=True)
REPO = WORK / 'learn'
# ⚠️ Chatterbox pins torch==2.6.0. Installing it into Kaggle's own environment downgrades torch under a
# torchvision built for a newer one ("operator torchvision::nms does not exist", and transformers'
# LlamaModel import dies with it). So it gets its OWN venv: its torch, no torchvision, nothing shared.
VENV = WORK / 'venv'; PY = VENV / 'bin' / 'python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
    subprocess.run([str(PY), '-m', 'pip', 'install', '-q', 'setuptools<81', 'chatterbox-tts', 'imageio-ffmpeg'], check=True)
print(subprocess.run([str(PY), '-c', "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set the accelerator'); print('torch', torch.__version__)"], capture_output=True, text=True).stdout)

In [ ]:
OUT = REPO / 'public/audio' / VOICE          # existing clips live here, so the script skips them
marker = WORK / 'started.marker'; marker.touch()
zip_base = ZIP_DIR / f'clips-{VOICE[:4]}-{time.strftime("%Y%m%d-%H%M")}'

def new_clips():
    return [p for p in OUT.glob('*.mp3') if p.stat().st_mtime > marker.stat().st_mtime and p.stat().st_size > 1024]

def rezip():
    stage = WORK / 'stage'; shutil.rmtree(stage, ignore_errors=True); stage.mkdir()
    files = new_clips()
    for p in files: shutil.copy2(p, stage / p.name)
    shutil.make_archive(str(zip_base), 'zip', stage)
    return len(files)

t0 = time.time(); chunks = 0; out_of_time = False; failures = 0
for CORPUS, BAND in PLAN:
    corpus = REPO / 'scripts' / f'.voice-corpus-{CORPUS}.json'
    while True:
        if time.time() - t0 > HOURS * 3600: out_of_time = True; break
        cmd = [str(PY), str(REPO / 'scripts/chatterbox-render.py'), '--voice', VOICE, '--corpus', str(corpus),
               '--out', str(OUT), '--limit', str(CHUNK)] + (['--band', BAND] if BAND else [])
        r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
        if '· 0 to render' in r.stdout: print(f'{CORPUS} {BAND}: complete'); break
        head = next((l for l in r.stdout.splitlines() if 'to render' in l), r.stdout[-300:])
        tail = [l for l in r.stdout.splitlines() if l.strip()[:1].isdigit()][-1:]
        chunks += 1
        print(f'[{CORPUS} {BAND}] chunk {chunks}: {head.strip()} | last: {tail[0].strip() if tail else "-"} | zipped {rezip()} new clips | {(time.time()-t0)/60:.0f} min', flush=True)
        if r.returncode != 0:
            failures += 1; print(r.stderr[-1500:])
            if failures >= 3: raise SystemExit('three chunks in a row failed — fix the error above before spending more GPU time')
            time.sleep(20)
        else: failures = 0
    if out_of_time: print('time budget reached — Run All again after merging the zip'); break
print('done —', rezip(), 'new clips in', str(zip_base) + '.zip')

Download the zip (Kaggle: right sidebar → Output; Colab: it is on Drive under `milo-voice/`), hand it over, and once it is merged, **Run All** again — the next run continues from where this one stopped, on whichever band is next in `PLAN`.